In [ ]:
{
 "nbformat": 4,
 "nbformat_minor": 5,
 "metadata": {
  "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
  "language_info": {"name": "python", "version": "3.10.0"}
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# 🐔 Poultry AI — Feature Engineering\n", "Notebook 02 | Deriving powerful predictors from raw contract data"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys; sys.path.insert(0, '..')\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from configs.config import RAW_CSV\n",
    "from src.preprocessing.data_loader import load_raw, clean\n",
    "\n",
    "sns.set_theme(style='whitegrid')\n",
    "df = clean(load_raw())\n",
    "print(f'Loaded {len(df)} records | {df.shape[1]} features')\n",
    "df.head(3)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 1. Engineered Features Overview"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "eng_features = [\n",
    "    'mortality_rate', 'feed_efficiency', 'revenue_per_bird',\n",
    "    'profit_per_bird', 'duration_efficiency', 'farm_score'\n",
    "]\n",
    "df[eng_features].describe().T.style.background_gradient(cmap='Greens')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 2. Mortality Rate Analysis"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# Distribution by risk label\n",
    "for label, color in [('Low','#22c55e'), ('Medium','#f59e0b'), ('High','#ef4444')]:\n",
    "    subset = df[df['risk_label'] == label]['mortality_rate']\n",
    "    axes[0].hist(subset, bins=30, alpha=0.6, label=label, color=color)\n",
    "axes[0].set_title('Mortality Rate by Risk Level')\n",
    "axes[0].set_xlabel('Mortality Rate')\n",
    "axes[0].legend()\n",
    "\n",
    "# Mortality vs Profit scatter\n",
    "colors = df['risk_label'].map({'Low':'#22c55e','Medium':'#f59e0b','High':'#ef4444'})\n",
    "axes[1].scatter(df['mortality_rate'], df['profit_per_bird'], \n",
    "                c=colors, alpha=0.4, s=15)\n",
    "axes[1].set_title('Mortality Rate vs Profit per Bird')\n",
    "axes[1].set_xlabel('Mortality Rate')\n",
    "axes[1].set_ylabel('Profit per Bird (₹)')\n",
    "axes[1].axvline(0.12, color='orange', linestyle='--', label='Medium threshold')\n",
    "axes[1].axvline(0.18, color='red',    linestyle='--', label='High threshold')\n",
    "axes[1].legend(fontsize=8)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 3. Feed Efficiency vs Profit"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, ax = plt.subplots(figsize=(10, 5))\n",
    "bins = pd.cut(df['feed_efficiency'], bins=5)\n",
    "grouped = df.groupby(bins)['estimated_profit'].mean()\n",
    "grouped.plot(kind='bar', ax=ax, color='#2563EB', edgecolor='white')\n",
    "ax.set_title('Avg Profit by Feed Efficiency Band', fontsize=13, fontweight='bold')\n",
    "ax.set_xlabel('Feed Efficiency (kg/bird)')\n",
    "ax.set_ylabel('Avg Estimated Profit (₹)')\n",
    "ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))\n",
    "plt.xticks(rotation=30)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 4. Farm Score Distribution"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "sns.histplot(df['farm_score'], bins=30, kde=True, ax=axes[0], color='#0f766e')\n",
    "axes[0].axvline(df['farm_score'].mean(), color='red', linestyle='--',\n",
    "                label=f'Mean: {df[\"farm_score\"].mean():.1f}')\n",
    "axes[0].set_title('Farm Score Distribution')\n",
    "axes[0].legend()\n",
    "\n",
    "sns.boxplot(data=df, x='contract_type', y='farm_score',\n",
    "            palette=['#2563EB','#0f766e','#7c3aed'], ax=axes[1])\n",
    "axes[1].set_title('Farm Score by Contract Type')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 5. Feature Importance (Quick RF check)"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.ensemble import RandomForestRegressor\n",
    "from sklearn.preprocessing import LabelEncoder\n",
    "from configs.config import ALL_FEATURES, CATEGORICAL_FEATURES, TARGET_PROFIT\n",
    "from src.preprocessing.data_loader import encode_categoricals, get_X_y\n",
    "\n",
    "X, y = get_X_y(df, TARGET_PROFIT)\n",
    "rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)\n",
    "rf.fit(X, y)\n",
    "\n",
    "fi = pd.Series(rf.feature_importances_, index=ALL_FEATURES).sort_values(ascending=True)\n",
    "fig, ax = plt.subplots(figsize=(10, 7))\n",
    "fi.tail(15).plot(kind='barh', ax=ax, color='#2563EB')\n",
    "ax.set_title('Top 15 Feature Importances (Random Forest)', fontsize=13, fontweight='bold')\n",
    "ax.set_xlabel('Importance')\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "print('Top 5 features:', fi.tail(5).index.tolist()[::-1])"
   ]
  }
 ]
}
